In [ ]:
from  datasets import load_dataset
import tqdm as notebook_tqdm
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset


In [ ]:
def tokenize(sentence):
    return sentence.split()

def standardize(text):
    text=text.lower()
    text=re.sub(r"[^\w\s]","",text) #remove puntuation
    return text

def causal_mask(size):
    mask=torch.triu(torch.ones((1,size,size)),diagonal=1).type(torch.int)
    return mask ==0

def encode(sentence, vocab):
    return [vocab.get(word, vocab["[unk]"]) for word in tokenize(sentence)]

class OscarNavDataset(Dataset):
    def __init__(self,ds,tokenizer_src,tokenizer_tgt,src_lang,tgt_lang,seq_len):
        super().__init__()
        self.seq_len =seq_len
        self.ds =ds
        self.tokenizer_src=tokenizer_src
        self.tokenizer_tgt=tokenizer_tgt
        self.src_lang=src_lang
        self.tgt_lang=tgt_lang
                      
        self.sos_token= encode("[SOS]", self.tokenizer_tgt)[0] #torch.tensor([tokenizer_tgt.token_to_id("[SOS]")],dtype=torch.int64)
        self.eos_token= encode("[EOS]", self.tokenizer_tgt)[0]#torch.tensor([tokenizer_tgt.token_to_id("[EOS]")],dtype=torch.int64)
        self.pad_token= encode("[PAD]", self.tokenizer_tgt)[0] #torch.tensor([tokenizer_tgt.token_to_id("[PAD]")],dtype=torch.int64)
        # print(self.pad_token)

    def __len__(self):
        return len(self.ds)
    
    def __getitem__(self, index):
        src_target_pair=self.ds[index]
        src_text = standardize(src_target_pair[self.src_lang])
        tgt_text = standardize(src_target_pair[self.tgt_lang])
        # src_text= src_target_pair[self.src_lang] #src_target_pair['translation'][self.src_lang]
        # tgt_text= src_target_pair[self.tgt_lang] #src_target_pair['translation'][self.tgt_lang]


        #Transform the text into tokens
        enc_input_tokens = encode(src_text, self.tokenizer_src) #self.tokenizer_src.encode(src_text).ids
        dec_input_tokens = encode(tgt_text, self.tokenizer_tgt) #self.tokenizer_tgt.encode(src_text).ids

        #Add sos,eos and padding to each sentence
        enc_num_padding_tokens=self.seq_len-len(enc_input_tokens)-2 #We will add <s> and </s>
        dec_num_padding_tokens=self.seq_len-len(dec_input_tokens)-1 #We will only add <s>, and </s> only on the label

        #Make sure the number of padding tokens is not negative. If it is then the sentence is too long
        if enc_num_padding_tokens <0 or dec_num_padding_tokens<0:
            raise ValueError("Sentence is too long")
        
        #Add <s> and </s> token
        encoder_input = torch.cat([
            torch.tensor([self.sos_token], dtype=torch.int64),#self.sos_token,
            torch.tensor(enc_input_tokens,dtype=torch.int64),
            torch.tensor([self.eos_token], dtype=torch.int64),#self.eos_token,
            torch.full((enc_num_padding_tokens,), self.pad_token, dtype=torch.int64) #torch.tensor([self.pad_token]*enc_num_padding_tokens,dtype=torch.int64)
        ],
        dim=0)

        #Add only <s> token
        decoder_input = torch.cat([
            torch.tensor([self.sos_token], dtype=torch.int64),#self.sos_token,
            torch.tensor(dec_input_tokens,dtype=torch.int64),
            torch.tensor([self.pad_token]*dec_num_padding_tokens,dtype=torch.int64)
        ],
        dim=0)

        #Add only </s> token
        label = torch.cat([
            torch.tensor(dec_input_tokens,dtype=torch.int64),
            torch.tensor([self.eos_token], dtype=torch.int64), #self.eos_token,
            torch.tensor([self.pad_token]*dec_num_padding_tokens,dtype=torch.int64),
            
        ],
        dim=0)


        #Check the size of the tensors
        assert encoder_input.size(0)==self.seq_len
        assert decoder_input.size(0)==self.seq_len
        assert label.size(0)==self.seq_len


        return{
            "encoder_input":encoder_input, #seq_len
            "decoder_input":decoder_input, #seq_len
            "encoder_mask":(encoder_input != self.pad_token).unsqueeze(0).unsqueeze(1).int(),#(encoder_input!=self.pad_token).unsqueeze(0).int(), #(1,1,seq_len)
            "decoder_mask": (decoder_input!=self.pad_token).unsqueeze(0).int() & causal_mask(decoder_input.size(0)),  #(1,seq_len) & (1,,seq_len,seq_len)
            "label":label,
            "src_text":src_text,
            "tgt_text":tgt_text
        }

In [ ]:
from pathlib import Path
def get_config():
    return {
        "batch_size": 8,
        "num_epochs": 20,
        "lr": 10**-4,
        "seq_len": 50,
        "d_model": 256,
        "datasource": 'opus_books',
        "lang_src": "eng",
        "lang_tgt": "spa",
        "model_folder": "weights",
        "model_basename": "tmodel_",
        "preload": "latest",
        "tokenizer_file": "tokenizer_{0}.json",
        "experiment_name": "runs/tmodel"
    }


def get_weights_file_path(config, epoch: str):
    model_folder = f"{config['datasource']}_{config['model_folder']}"
    model_filename = f"{config['model_basename']}{epoch}.pt"
    return str(Path('.') / model_folder / model_filename)

# Find the latest weights file in the weights folder
def latest_weights_file_path(config):
    model_folder = f"{config['datasource']}_{config['model_folder']}"
    model_filename = f"{config['model_basename']}*"
    weights_files = list(Path(model_folder).glob(model_filename))
    if len(weights_files) == 0:
        return None
    weights_files.sort()
    return str(weights_files[-1])

In [ ]:
import torch
import torch.nn as nn
import math

class InputEmbedding(nn.Module):
    def __init__(self, d_model:int , vocab_size:int):
        super().__init__()
        self.d_model=d_model
        self.vocab_size=vocab_size
        self.embedding = nn.Embedding(vocab_size,d_model)
    
    def forward(self,x):
        return self.embedding(x)*math.sqrt(self.d_model)
    
class PositionalEncoding(nn.Module):
    def __init__(self, d_model:int, seq_len:int,dropout:float):   #seq_len is the maximum length of the sentence and dropout is to reduce overfit
        super().__init__()
        self.d_model=d_model
        self.seq_len=seq_len
        self.dropout = nn.Dropout(dropout)

        #Create a matrix of shape (seqlen, d_model)
        pe = torch.zeros(seq_len,d_model)

        #Create a vector of shape (seq_len,1)
        position= torch.arange(0,seq_len,dtype=torch.float).unsqueeze(1)
        div_term= torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))

        #Apply Sin to Even poisition and Cos to odd position
        pe[:,0::2]=torch.sin(position*div_term)
        pe[:,1::2]=torch.cos(position*div_term)


        pe =pe.unsqueeze(0) #(1seq_len ,d_model)

        self.register_buffer('pe',pe)

    def forward(self,x):
        x+= (self.pe[:,:x.shape[1],:]).requires_grad_(False)
        return self.dropout(x)
    

class LayerNormalization(nn.Module):
    def __init__(self, eps:float=10**-6):
        super().__init__()
        self.eps=eps
        self.alpa=nn.Parameter(torch.ones(1)) #multiplied
        self.bias=nn.Parameter(torch.ones(1)) #Added

    def forward(self,x):
        mean=x.mean(dim=-1,keepdim=True)
        std =x.std(dim=-1,keepdim=True)
        return self.alpa *(x-mean)/(std+self.eps) +self.bias
    
class FeedForwardBlock(nn.Module):
    def __init__(self, d_model:int, d_ff:int,dropout:float):
        super().__init__()
        self.linear_1=nn.Linear(d_model,d_ff)
        self.dropout=nn.Dropout(dropout)
        self.linear_2=nn.Linear(d_ff,d_model)

    def forward(self,x):
        # (Batch,Seq_Len, d_model)--> (Batch,Seq_Len,d_ff) -->(Batch,Seq_Len,d_model)
        return self.linear_2(self.dropout(self.linear_1(x)))

class MultiHeadAttentionBlock(nn.Module):

    def __init__(self,d_model:int,h:int,dropout:float):
        super().__init__()
        self.d_model= d_model #Embedding vector size
        self.h =h #Number of heads

        #make sured_model is divisible by h
        assert d_model %h ==0, 'd_model is not divisible by h'

        self.d_k = d_model //h #Dimension of vector seen by each head
        self.w_q =nn.Linear(d_model,d_model,bias=False) #Wq
        self.w_k =nn.Linear(d_model,d_model,bias=False) #Wk
        self.w_v =nn.Linear(d_model,d_model,bias=False) #Wv
        self.w_o =nn.Linear(d_model,d_model,bias=False) #Wo
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query,key,value,mask,dropout: nn.Dropout):

        d_k=query.shape[-1]
        #Just apply the formula from the paper
        #(batch, h,seq_len,d_k) --> (batch,h, seq_len,seq_len)
        attention_scores = (query@key.transpose(-2,-1))/math.sqrt(d_k)

        if mask is not None:
            #Write a very low value (indicating -inf) to the positions where mask ==0
            attention_scores.masked_fill_(mask==0,-1e9)
        
        attention_scores = attention_scores.softmax(dim=-1) #(batch ,h, seq_len,seq_len) Apply softmax

        if dropout is not None:
            attention_scores = dropout(attention_scores)
        #(batch,h,seq_len,seq_len) --> (batch,h,seq_len,d_k)
        #return attention scores which can be used for visualization

        return(attention_scores @ value),attention_scores
    
    def forward (self,q,k,v,mask):
        query = self.w_q(q) # ( batch, seq_len,d_model) --> (batch,seq_len,d_model)
        key =self.w_k(k) # ( batch, seq_len,d_model) --> (batch,seq_len,d_model)
        value =self.w_v(v) # ( batch, seq_len,d_model) --> (batch,seq_len,d_model)

        #(batch,seq_len,d_model) --> (batch,seq_len,h,d_k) -->(batch,h,seq_len,d_k)
        query = query.view(query.shape[0],query.shape[1],self.h,self.d_k).transpose(1,2)
        key = key.view(key.shape[0],key.shape[1],self.h,self.d_k).transpose(1,2)
        value = value.view(value.shape[0],value.shape[1],self.h,self.d_k).transpose(1,2)


        #Calculate attention
        x,self.attention_score = MultiHeadAttentionBlock.attention(query,key,value,mask,self.dropout)

        #Combine all the heads together
        #(batch,h,seq_len,d_k) -- > (batch,seq_len,h,d_k) --> (batch,seq_len, d_model)
        x =x.transpose(1,2).contiguous().view(x.shape[0],-1,self.h*self.d_k)

        #Multiply by Wo
        #(batch,seq_len,d_model) --> (batch,seq_len,d_model)
        return self.w_o(x)


class ResidualConnection (nn.Module):
    def __init__(self, features:int,dropout:float):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm=LayerNormalization(features)

    def forward(self,x,sublayer):        
        return x+self.dropout(sublayer(self.norm(x)))
    
    
    

class EncoderBlock(nn.Module):

    def __init__(self,features:int, self_attention_block:MultiHeadAttentionBlock,feed_forward_block:FeedForwardBlock,dropout:float):
        super().__init__()
        self.self_attention_block = self_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connection = nn.ModuleList([ResidualConnection(features,dropout) for _ in range(2)])

    def forward(self,x,src_mask):
        x = self.residual_connection[0](x, lambda x: self.self_attention_block(x,x,x,src_mask))
        x = self.residual_connection[1](x,self.feed_forward_block)
        return x

class Encoder(nn.Module):
    def __init__(self,features:int, layers:nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(features)
    
    def forward(self,x,mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)
    

class DecoderBlock(nn.Module):
    def __init__(self, features:int,self_attention_block:MultiHeadAttentionBlock,cross_attention_block:MultiHeadAttentionBlock,feed_forward_block:FeedForwardBlock,drop_out:float):
        super().__init__()
        self.self_attention_block = self_attention_block
        self.cross_attention_block=cross_attention_block
        self.feed_forward_block=feed_forward_block  
        self.residual_connections=nn.ModuleList([ResidualConnection(features,drop_out) for _ in range(3)])

    def forward(self,x,encoder_output,src_mask,tgt_mask):
        x =self.residual_connections[0](x,lambda x:self.self_attention_block(x,x,x,tgt_mask))
        x= self.residual_connections[1](x,lambda x : self.cross_attention_block(x,encoder_output,encoder_output,src_mask))
        x= self.residual_connections[2](x,self.feed_forward_block)
        return x
    
class Decoder(nn.Module):
    def __init__(self, features:int,layers:nn.ModuleList):
        super().__init__()
        self.layers= layers
        self.norm = LayerNormalization(features)

    def forward(self,x,encoder_output,src_mask,tgt_mask):
        for layer in self.layers:
            x = layer(x,encoder_output,src_mask,tgt_mask)
        return self.norm(x)
    
class ProjectionLayer(nn.Module):
    def __init__(self,d_model,vocab_size):
        super().__init__()
        self.proj = nn.Linear(d_model,vocab_size)

    def forward (self,x):
        return self.proj(x)
    
class Transformer(nn.Module):
     def __init__(self, encoder:Encoder,decoder:Decoder,src_embed:InputEmbedding,tgt_embed:InputEmbedding,src_pos:PositionalEncoding,tgt_pos:PositionalEncoding,projection_layer:ProjectionLayer):
         super().__init__()
         self.encoder=encoder
         self.decoder=decoder
         self.src_embed=src_embed
         self.tgt_embed=tgt_embed
         self.src_pos=src_pos
         self.tgt_pos=tgt_pos
         self.projection_layer=projection_layer

        
     def encode(self,src,src_mask):
        #(batch,seq_len,d_model)
        src=self.src_embed(src)
        src=self.src_pos(src)
        return (self.encoder(src,src_mask))
    
     def decode (self,encodr_output:torch.Tensor,src_mask:torch.Tensor,tgt:torch.Tensor,tgt_mask:torch.Tensor):
         #(batch,seq_len,d_model)
         tgt = self.tgt_embed(tgt)
         tgt = self.tgt_pos(tgt)
         return self.decoder(tgt,encodr_output,src_mask,tgt_mask)
     
     def project(self,x):
         #(batch,seq_len,vocab_size)
         return self.projection_layer(x)
     

def build_transformer(src_vocab_size:int,tgt_vocab_size:int,src_seq_len:int,tgt_seq_len:int,d_model:int=512,N:int=6,h:int=4,dropout:float=0.1,d_ff:int=2048):
    #Create the Embedding Layer
    src_embed= InputEmbedding(d_model,src_vocab_size)
    tgt_embed= InputEmbedding(d_model,tgt_vocab_size)

    #Create the positional encoding layers
    src_pos=PositionalEncoding(d_model,src_seq_len,dropout)
    tgt_pos=PositionalEncoding(d_model,tgt_seq_len,dropout)

    #Create encoder block
    encoder_blocks=[]
    for _ in range(N):
        encoder_self_attention_block=MultiHeadAttentionBlock(d_model,h,dropout)
        feed_forward_block = FeedForwardBlock(d_model,d_ff,dropout)
        encoder_block=EncoderBlock(d_model,encoder_self_attention_block,feed_forward_block,dropout)
        encoder_blocks.append(encoder_block)

    decoder_blocks=[]
    #Create the decoder block
    for _ in range(N):
        decoder_self_attention_block=MultiHeadAttentionBlock(d_model,h,dropout)
        decoder_cross_attention_block=MultiHeadAttentionBlock(d_model,h,dropout)
        feed_forward_block = FeedForwardBlock(d_model,d_ff,dropout)
        decoder_block=DecoderBlock(d_model,decoder_self_attention_block,decoder_cross_attention_block,feed_forward_block,dropout)
        decoder_blocks.append(decoder_block)

    #Create the encoder and decoder
    encoder = Encoder(d_model,nn.ModuleList(encoder_blocks))
    decoder = Decoder(d_model,nn.ModuleList(decoder_blocks))

    #Create the projection Layer
    projection_layer = ProjectionLayer(d_model,tgt_vocab_size)


    #Create the Transformer
    transformer = Transformer(encoder,decoder,src_embed,tgt_embed,src_pos,tgt_pos,projection_layer)


    #Initialise the parameters
    for p in transformer.parameters():
        if p.dim()>1:
            nn.init.xavier_uniform_(p)
    
    return transformer


In [ ]:
# from model import build_transformer
# from config import get_config, get_weights_file_path, latest_weights_file_path
# from  datasets import load_dataset
# from dataset import OscarNavDataset,encode,causal_mask

!pip install torchmetrics
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchmetrics
from torch.utils.tensorboard import SummaryWriter

import re
import warnings
from tqdm import tqdm
from collections import Counter
from pathlib import Path
import os



dataset =load_dataset("OscarNav/spa-eng")

def greedy_decode(model,source,source_mask,tokenizer_src,tokenizer_tgt,max_len,device):
    sos_idx = encode("[SOS]", tokenizer_tgt)[0]
    eos_idx = encode("[EOS]", tokenizer_tgt)[0]

    #Precompute the encoder output nad reuse it for every step
    encoder_output= model.encode(source,source_mask)

    #Initialize the decoder input with sos token
    decoder_input = torch.empty(1,1).fill_(sos_idx).type_as(source).to(device)

    while True:
        if decoder_input.size(1)== max_len:
            break

        #build mask for target
        decoder_mask=causal_mask(decoder_input.size(1)).type_as(source_mask).to(device)

        #Calculate Output
        out = model.decode(encoder_output,source_mask,decoder_input,decoder_mask)

        #get next token
        prob = model.project(out[:,-1])
        _,next_word = torch.max(prob,dim=1)
        next_word_id = next_word.item()
        next_word_tensor = torch.tensor([[next_word_id]], dtype=torch.long).to(device)
        decoder_input = torch.cat([decoder_input, next_word_tensor], dim=1)
        # decoder_input = torch.cat([decoder_input,torch.empty(1,1).type_as(source).fill_(next_word.item().to(device))],dim=1)

        if next_word ==eos_idx:
            break
    
    return decoder_input.squeeze(0)

def run_validation(model,validation_ds,tokenizer_src,tokenizer_tgt,max_len,device,print_msg,global_step,writer,num_examples=2):

    model.eval()
    count =0
    id_to_word = {v: k for k, v in tokenizer_tgt.items()}

    source_texts =[]
    expected =[]
    predicted =[]

    try:
        # get the console window width
        with os.popen('stty size', 'r') as console:
            _, console_width = console.read().split()
            console_width = int(console_width)
    except:
        # If we can't get the console width, use 80 as default
        console_width = 80
    
    with torch.no_grad():
        for batch in validation_ds:
            count +=1
            encoder_input = batch["encoder_input"].to(device) #(b,seq_len)
            encoder_mask  = batch["encoder_mask"].to(device) #(b,1,1,seq_len)

            #check that the batch size is 1

            assert encoder_input.size(0)==1,"Batch size must be 1 for validation"


            model_out = greedy_decode( model, encoder_input,encoder_mask,tokenizer_src,tokenizer_tgt,max_len,device)

            source_text=batch["src_text"][0]
            target_text=batch["tgt_text"][0]
            model_out_text = decode(model_out.detach().cpu().numpy(), id_to_word)
            source_texts.append(source_text)
            expected.append(target_text)
            predicted.append(model_out_text)

            #Print the source, target and model output
            print_msg('-'*console_width)
            print_msg(f"{f'SOURCE: ':>12}{source_text}")
            print_msg(f"{f'TARGET: ':>12}{target_text}")
            print_msg(f"{f'PREDICTED: ':>12}{model_out_text}")

            if count == num_examples:
                print_msg("-"*console_width)
                break
        
    if writer:
        #Evaulate the character error rate
        #Compute  the char error rate
        mertic = torchmetrics.CharErrorRate()
        cer = mertic(predicted,expected)
        writer.add_scalar("validation cer", cer ,global_step)
        writer.flush()

        #Compute the word error rate
        metric= torchmetrics.WordErrorRate()
        wer= mertic(predicted,expected)
        writer.add_scalar('validation wer',wer,global_step)
        writer.flush()

        #Compute BLEU metric
        metric=torchmetrics.BLEUScore()
        bleu=metric(predicted,expected)
        writer.add_scalar('validation BLEU',bleu,global_step)
        writer.flush()




def tokenize(sentence):
    return sentence.split()

def decode(ids, id_to_word):
    words = []
    for i in ids:
        word = id_to_word.get(i, "[unk]")
        if word == "[EOS]":
            break
        if word not in ["[PAD]", "[SOS]"]:
            words.append(word)
    return " ".join(words)




def build_vocab(sentences, max_vocab_size=15000):
    counter = Counter()
    
    for sentence in sentences:
        counter.update(tokenize(sentence))
    
    special_tokens = ["[PAD]", "[unk]", "[SOS]", "[EOS]"]
    
    most_common = counter.most_common(max_vocab_size - len(special_tokens))
    
    vocab = {token: idx for idx, token in enumerate(special_tokens)}
    
    for word, _ in most_common:
        vocab[word] = len(vocab)
    
    return vocab


def get_ds(config):
    dataset =load_dataset("OscarNav/spa-eng")

    #Use only the  first 50,000 setance pairs
    dataset=dataset['train'].select(range(50000))
    english_sentence=dataset[config['lang_src']]
    spanish_sentence=dataset[config['lang_tgt']]

    english_sentence=[standardize(s) for s in english_sentence]
    spanish_sentence=[standardize(s) for s in spanish_sentence]

    #Build Tokenizers
    src_vocab = build_vocab(english_sentence, 15000)
    tgt_vocab = build_vocab(spanish_sentence, 15000)

    #Keep 90% training and 10% validation
    train_ds_size= int(0.9* len(dataset))
    val_ds_size=len(dataset)-train_ds_size

    train_ds_raw,val_ds_raw =random_split(dataset,[train_ds_size,val_ds_size])

    train_ds = OscarNavDataset(train_ds_raw,src_vocab,tgt_vocab,config["lang_src"],config['lang_tgt'],config['seq_len'])
    val_ds = OscarNavDataset(val_ds_raw,src_vocab,tgt_vocab,config["lang_src"],config['lang_tgt'],config['seq_len'])

    train_dataloader = DataLoader(train_ds,batch_size=config['batch_size'],shuffle=True)
    val_dataloader = DataLoader(val_ds,batch_size=1,shuffle=True)

    return train_dataloader,val_dataloader,src_vocab,tgt_vocab


def get_model(config,vocab_src_len,vocab_tgt_len):
    model = build_transformer(vocab_src_len,vocab_tgt_len,config['seq_len'],config['seq_len'],config['d_model'])
    return (model)


def train_model(config):
    #Define the device
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.has_mps or torch.backends.mps.is_available() else 'cpu'
    print("Using device:",device)

    if (device =="cuda"):
        print(f"Device Name:{torch.cuda.get_device_name(device.index)}")
        print(f"Device Memory:{torch.cuda.get_device_properties(device.index).total_memory/1024*3} GB")
    elif (device=='mps'):
        print(f"Device name: <mps")

    else:
        print("NOTE: If you have a GPU, consider using it for training.")

    device = torch.device(device)

    # Make sure the weights folder exists
    Path(f"{config['datasource']}_{config['model_folder']}").mkdir(parents=True, exist_ok=True) 

    train_loader,val_dataloader,tokenizer_src, tokenizer_tgt = get_ds(config)

    model = get_model(config,len(tokenizer_src),len(tokenizer_tgt)).to(device)

    #Tensor Board
    writer = SummaryWriter(config['experiment_name'])
    
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'], eps=1e-9)
    
    #If the user specified a model to preload beore training
    initial_epoch = 0
    global_step = 0
    preload = config['preload']
    model_filename = latest_weights_file_path(config) if preload == 'latest' else get_weights_file_path(config, preload) if preload else None
    if model_filename:
        print(f'Preloading model {model_filename}')
        state = torch.load(model_filename)
        model.load_state_dict(state['model_state_dict'])
        initial_epoch = state['epoch'] + 1
        optimizer.load_state_dict(state['optimizer_state_dict'])
        global_step = state['global_step']
    else:
        print('No model to preload, starting from scratch')

    
    loss_fn=nn.CrossEntropyLoss(ignore_index=tokenizer_tgt["[PAD]"],label_smoothing=0.1).to(device)

    for epoch in range(initial_epoch,config['num_epochs']):
        torch.cuda.empty_cache()
        model.train()
        batch_iterator = tqdm(train_loader,desc=f"Processing Epoch {epoch:02d}")
        train_losses=[]

        for batch in batch_iterator:
            encoder_input = batch['encoder_input'].to(device) # (b,seq_len)
            decoder_input = batch['decoder_input'].to(device) # (b,seq_len)
            encoder_mask  = batch['encoder_mask'].to(device) # (b,seq_len)
            decoder_mask  = batch['decoder_mask'].to(device) # (b,seq_len)

            #Run the Tensor through encoder Decoder and the projection Layer
            encoder_output = model.encode(encoder_input,encoder_mask) # (B, seq_len, d_model)
            decoder_output = model.decode(encoder_output,encoder_mask,decoder_input,decoder_mask)  # (B, seq_len, d_model)
            proj_output = model.project(decoder_output) # (B, seq_len, vocab_size)

            #Compare the output with the label
            label = batch['label'].to(device) # (B, seq_len)

            #Compute the loss using simple cross entropy
            loss =loss_fn(proj_output.view(-1,len(tokenizer_tgt)),label.view(-1))
            train_losses.append(loss.item())
            batch_iterator.set_postfix({"loss": f"{loss.item():6.3f}"})

             # Log the loss
            writer.add_scalar('train loss', loss.item(), global_step)
            writer.flush()

            #back porpagate the loss
            loss.backward()

            # Update the weights
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1

        # Run validation at the end of every epoch
        run_validation(model, val_dataloader, tokenizer_src, tokenizer_tgt, config['seq_len'], device, lambda msg: batch_iterator.write(msg), global_step, writer)
        
        # Save the model at the end of every epoch
        model_filename = get_weights_file_path(config, f"{epoch:02d}")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'global_step': global_step
        }, model_filename)



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
config = get_config()
train_model(config)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(train_losses)
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.grid(True)
plt.show()